# Fastest time solution

Import libraries

In [ ]:
import os
import folium
import json
import numpy as np
import pandas as pd
import branca.colormap as cm
import matplotlib.pyplot as plt

In [ ]:
from src.matrix_maker import create_time_matrix, create_distance_matrix, create_elevation_matrix, add_dummy_nodes
from src.cycling_model import Cyclist
from src.tsp import solve_tsp

Custom colours, use default colourmaps instead

In [ ]:
from pyush import MyColours
cmap = MyColours().specified_colour_ramp(11).hex[::-1]
mcol = MyColours()[0]

In [ ]:
# cmap = cm.LinearColormap(["Green", "Red"])
# mcol = "blue"

## Cyclist parameters

Setting model for cyclist speed, given assumptions, default parameters are:
```
power_output=150              # watts 
weight=70                     # Of cyclist in kg
bike_weight=10                # weight of bike in kg 
drive_train_loss=0.02         # loss from drive train, 
                              # 0 = no loss, 1 = total loss
area=0.509                    # Front area of cyclist m^2
air_density=1.22601           # kg/m^3
cda=0.4                       # coef of air drag
coef_rolling_resistance=0.005 # coef of rolling risistance

max_speed=45                  # maximum speed of rider in km/hr - 
                              # stops huge speeds downhill
                              
min_speed=10                  # minimum speed of rider in km/hr - 
                              # stops crawling up hills

corner_time_penalty=5         # number of seconds added to do a 180deg turn
corner_coef_a=5               # Coeficient 
corner_coef_b=2.5 * pi        # Coef
```

Fitting the parameters to give reasonable speeds

In [ ]:
cyclist_params = {
    "power_output": 100,
    "weight": 65,
    "bike_weight": 12,
    "max_speed": 40,
    "min_speed": 8
}

cyclist = Cyclist(**cyclist_params)

In [ ]:
# Speed given road gradient
gradient = np.arange(-5, 15)
speed_hill = list()
for grad in gradient:
    speed_hill.append(cyclist.calculate_speed(np.array([0, 0, 0]), np.array([100, 0, grad]), xy_coords=True) * 3.6)

# speed given head/tail wind 
wind_speed = np.arange(-10, 10)
speed_wind = list()
for wind in wind_speed:
    speed_wind.append(
        cyclist.calculate_speed(
            np.array([0, 0, 0]), 
            np.array([100, 0, 0]), 
            wind_direction=90, 
            wind_speed=wind,
            xy_coords=True
        ) * 3.6
    )


The cyclist speed is shown on the y-axis, with gradient and wind speed on the x-axis. Note that a negative wind is a tail wind and positive is a head wind.

The max and min speeds are caped. This can be altered, but is done to prevent huge speeds being achieved down hill, and people crawling up hills at much slower than walking speed. 

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(gradient, np.array(speed_hill), color=mcol)
ax[1].plot(wind_speed*3.6, np.array(speed_wind), color=mcol)

ax[0].set_xlabel("Gradient (%)")
ax[0].set_ylabel("Speed (km/h)")

ax[1].set_xlabel("Wind Speed (km/h)")
ax[1].set_ylabel("Speed (km/h)")
plt.show()

Cornering penalty

In [ ]:
p1 = np.array(
    [[0, 0, 0],
    [1, 0, 0]]
)

theta = np.linspace(0, np.pi, 100)
penilty = list()
for t in theta:
    x, y = np.cos(t), np.sin(t)
    p2 = np.array(
            [[1, 0, 0],
            [1 + np.cos(t), 0 + np.sin(t), 0]]
        )

    penilty.append(cyclist.cornering_time(p1, p2, xy_coords=True))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta * 180 / np.pi, penilty, color=mcol)

ax.set_xlabel("Degree of turn")
ax.set_ylabel("Added time as penilty (sec)")
plt.show()

## Solve problem to find route

Builds the time matrix, adding a dummy node for begining/ending point

In [ ]:
run_name = "Best_wind"

parameters = {
    "wind_speed": 20 / 3.6,
    "wind_direction": 213.0,
    "corner_penilty": True
}

In [ ]:
locs_time, mat_time = create_time_matrix(
    wind_speed=parameters["wind_speed"], 
    wind_direction=parameters["wind_direction"], 
    corner_penilty=parameters["corner_penilty"],
    **cyclist_params
)
locs_dist, mat_dist = create_distance_matrix(**cyclist_params)
locs_ele, mat_ele = create_elevation_matrix()

# Un comment the one you want to minimise

# Time
dummy_locs, dummy_mat = add_dummy_nodes(locs_time, mat_time)

# Distance
# dummy_locs, dummy_mat = add_dummy_nodes(locs_dist, mat_dist)

# Elevation
# dummy_locs, dummy_mat = add_dummy_nodes(locs_ele, mat_ele)

In [ ]:
solution_idx = solve_tsp(dummy_mat, 0)

Print out of route

In [ ]:
data_list = list()
data_list.append([dummy_locs[solution_idx['route'][1]].replace("_", " ").capitalize(), 0, 0, 0])

prev_idx = solution_idx['route'][1]

for i in solution_idx['route'][2:-1]:
    data_list.append([
        dummy_locs[i].replace("_", " ").capitalize(), 
        int(mat_time[prev_idx-1][i-1]),
        int(mat_dist[prev_idx-1][i-1]),
        int(mat_ele[prev_idx-1][i-1]),
        int(mat_ele[i-1][prev_idx-1])
    ])
    prev_idx = i

data_df = pd.DataFrame(data_list, columns=["Location", "Time (s)", "Distance (m)", "Elevation (m)", "Descent (m)"]) 
data_df["Ave Speed (km/h)"] = data_df["Distance (m)"] / data_df["Time (s)"] * 3.6
data_df["Cum. Time (min)"] = data_df["Time (s)"].cumsum() / 60
data_df["Cum. Dist (km)"] = data_df["Distance (m)"].cumsum() / 1000
data_df["Cum. Elev (m)"] = data_df["Elevation (m)"].cumsum()

data_df.to_csv("outputs/Results/" + run_name + ".csv")

In [ ]:
data_df

## Map solution

In [ ]:
cafe_locs = []
route = []

for p1, p2 in zip(solution_idx['route'][:-1], solution_idx['route'][1:]):
    if p1 != 0 and p2 != 0:
        loc1 = dummy_locs[p1]
        loc2 = dummy_locs[p2]
        file_name = "routes/" + loc1 + "-" + loc2 + ".json"
        file_name_rev = "routes/" + loc2 + "-" + loc1 + ".json"
        if os.path.exists(file_name):
            # Read the file and get points
            with open(file_name, "r") as f:
                data = json.load(f)
            route.append(data["paths"][0]["points"]["coordinates"])
            cafe_locs.append(data["paths"][0]["points"]["coordinates"][0])
        elif os.path.exists(file_name_rev):
            with open(file_name_rev, "r") as f:
                data = json.load(f)
            route.append(data["paths"][0]["points"]["coordinates"][::-1])
            cafe_locs.append(data["paths"][0]["points"]["coordinates"][-1])
        else:
            print("missing:" + loc1 + "-" + loc2)  

route = np.vstack(route)
# Add last cafe point
cafe_locs.append(route[-1])  

In [ ]:
spacer = 3

route_points = route[..., :2][::spacer, ::-1]
elev = route[::spacer, 2]

In [ ]:
# Create the base map
my_map = folium.Map(location=route_points[0], zoom_start=12, tiles="OpenStreetMap")

# Create a colormap (e.g., Linear from Green -> Yellow -> Red)
colormap = cm.LinearColormap(cmap, vmin=0, vmax=350)

# Draw the route segment by segment
for i in range(len(elev) - 1):
    point_a = route_points[i]
    point_b = route_points[i+1]
    
    # Coordinates for the segment
    segment_coords = [point_a, point_b]
    
    # Calculate average elevation for this segment to determine color
    avg_elevation = (elev[i] + elev[i+1]) / 2
    segment_color = colormap(avg_elevation)
    
    # Add segments to map
    folium.PolyLine(
        locations=segment_coords, 
        color=segment_color, 
        weight=5, 
        opacity=0.9,
        popup=f"Avg Elev: {int(avg_elevation)}m"
    ).add_to(my_map)

# Add Cafe locations
index_num = 0
for loc, name in zip(cafe_locs, solution_idx['route'][1:-1]):
    cl = loc[:2][::-1]
    folium.Marker(
        cl, 
        popup=str(index_num) +": " +  dummy_locs[name].replace("_", " ").capitalize(),
        icon=folium.Icon(color='gray', icon="coffee", prefix="fa")
    ).add_to(my_map)
    index_num += 1

# Add the colormap legend to the map
colormap.caption = 'Elevation (meters)'
colormap.add_to(my_map)

# Save map
my_map.save("outputs/Maps/" + run_name + ".html")

In [ ]:
pars = {
    "cyclist": cyclist.__dict__,
    "model": parameters
}

In [ ]:
with open("outputs/Results/" + run_name + "_params.json", "w") as f:
    json.dump(pars, f)